In [ ]:
import sys
!{sys.executable} -m pip install dill --user

In [ ]:
# Monte Carlo k-NN Optimization for Topological Data Analysis - Modified Version
# Run this in Jupyter Notebook

import os
import dill
import numpy as np
import random
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt
from collections import Counter
import pickle
from tqdm import tqdm

class MonteCarloKNNOptimizer:
    def __init__(self, data_root='${TDL_ROOT_DIR}/results/D1', true_b0=9):
        self.data_root = data_root
        self.true_b0 = true_b0
        self.train_data = None
        self.test_data = None
        self.full_dataset = None
        
    def load_data(self):
        """Load the synthetic training and test datasets"""
        pkl_path = os.path.join(self.data_root, 'train_test_split.pkl')
        
        print(f"Loading data from: {pkl_path}")
        
        try:
            with open(pkl_path, 'rb') as f:
                data_split = dill.load(f)
            
            print(f"Loaded data type: {type(data_split)}")
            print(f"Data structure: {data_split}")
            
            # Handle different data structures
            if hasattr(data_split, 'train_dataset'):
                # Object with attributes
                self.train_data = data_split.train_dataset
                self.test_data = data_split.test_dataset
            elif isinstance(data_split, (tuple, list)) and len(data_split) >= 2:
                # Tuple/list format: (train_data, test_data)
                self.train_data = data_split[0]
                self.test_data = data_split[1]
                print(f"Extracted from tuple/list format")
            elif isinstance(data_split, dict):
                # Dictionary format
                if 'train_dataset' in data_split:
                    self.train_data = data_split['train_dataset']
                    self.test_data = data_split['test_dataset']
                elif 'train' in data_split:
                    self.train_data = data_split['train']
                    self.test_data = data_split['test']
                else:
                    # Print available keys to help debug
                    print(f"Dictionary keys: {list(data_split.keys())}")
                    raise ValueError("Could not find train/test data in dictionary")
            else:
                print(f"Unexpected data format. Type: {type(data_split)}")
                if hasattr(data_split, '__dict__'):
                    print(f"Object attributes: {list(data_split.__dict__.keys())}")
                raise ValueError("Unknown data format")
            
            print(f"Successfully loaded data:")
            print(f"  Train dataset type: {type(self.train_data)}")
            print(f"  Test dataset type: {type(self.test_data)}")
            
            # Try to get shape information
            if hasattr(self.train_data, '__len__'):
                print(f"  Train dataset length: {len(self.train_data)}")
            if hasattr(self.test_data, '__len__'):
                print(f"  Test dataset length: {len(self.test_data)}")
                
            # If it's a tuple/list, let's also inspect the first few elements
            if isinstance(data_split, (tuple, list)):
                print(f"  Total elements in loaded structure: {len(data_split)}")
                for i, elem in enumerate(data_split[:3]):  # Show first 3 elements
                    print(f"  Element {i}: type={type(elem)}, shape={getattr(elem, 'shape', 'no shape attr')}")
                
        except Exception as e:
            print(f"Error loading data: {e}")
            raise
    
    def extract_data_points(self, dataset, max_batches=None):
        """Extract data points from dataset (assuming it's a DataLoader or similar)"""
        rows = []
        batch_count = 0
        
        try:
            # If dataset is iterable (like DataLoader)
            for batch_data in dataset:
                if isinstance(batch_data, (list, tuple)):
                    xb = batch_data[0]  # First element should be the data
                else:
                    xb = batch_data
                
                # Convert to numpy if it's a tensor
                if hasattr(xb, 'detach'):
                    x_np = xb.detach().cpu().numpy()
                else:
                    x_np = np.array(xb)
                
                # Reshape to 2D if needed
                if x_np.ndim > 2:
                    x_np = x_np.reshape(x_np.shape[0], -1)
                elif x_np.ndim == 1:
                    x_np = x_np.reshape(1, -1)
                
                rows.append(x_np)
                batch_count += 1
                
                if max_batches and batch_count >= max_batches:
                    break
                    
        except Exception as e:
            print(f"Error extracting data points: {e}")
            # If dataset is already a numpy array or similar
            if hasattr(dataset, 'shape'):
                return dataset.reshape(dataset.shape[0], -1)
            raise
        
        if rows:
            data_points = np.vstack(rows)
            print(f"Extracted {data_points.shape[0]} data points with {data_points.shape[1]} features")
            return data_points
        else:
            raise ValueError("No data points could be extracted from dataset")
    
    def find_all_k_for_subset(self, subset, max_k=100):
        """Find ALL k values that work for a given subset - MODIFIED VERSION"""
        valid_ks = []
        
        for k in range(1, max_k + 1):
            try:
                # Can't have more neighbors than data points
                if k >= len(subset):
                    break
                
                # Create k-NN graph
                neighbors = NearestNeighbors(n_neighbors=k+1).fit(subset)  # +1 because includes self
                graph = neighbors.kneighbors_graph(subset, mode='connectivity')
                
                # Remove self-connections and make undirected
                graph.setdiag(0)
                graph_undirected = graph.maximum(graph.T)
                graph_undirected.eliminate_zeros()
                
                # Count connected components
                n_components, _ = connected_components(csgraph=graph_undirected, directed=False)
                
                if n_components == self.true_b0:
                    valid_ks.append(k)  # Add to list instead of returning immediately
                    
            except Exception as e:
                continue
        
        return valid_ks  # Return all valid k values
    
    def monte_carlo_k_optimization_all(self, n_trials=1000, subset_fraction=0.25, max_k=100, use_train_data=True):
        """Run Monte Carlo simulation to find ALL optimal k values and save working subsets"""
        print(f"\nRunning Monte Carlo optimization (finding ALL k values):")
        print(f"  Number of trials: {n_trials}")
        print(f"  Using {subset_fraction*100}% of data each trial")
        print(f"  Target B0: {self.true_b0}")
        print(f"  Max k to test: {max_k}")
        
        # Choose dataset and extract all points
        dataset = self.train_data if use_train_data else self.test_data
        self.full_dataset = self.extract_data_points(dataset)
        
        # Calculate subset size as 25% of total data
        total_points = len(self.full_dataset)
        subset_size = int(total_points * subset_fraction)
        
        print(f"  Total data points: {total_points}")
        print(f"  Subset size (25%): {subset_size}")
        
        if subset_size < 50:  # Minimum reasonable subset size
            print(f"Warning: Subset size {subset_size} is very small. Consider using more data.")
        
        all_k_results = []  # Store all results: (trial_idx, valid_ks, subset)
        all_successful_ks = []  # Flat list of all k values that worked
        failed_trials = 0
        
        # Use tqdm for progress bar in Jupyter
        for trial in tqdm(range(n_trials), desc="Monte Carlo trials"):
            try:
                # Generate random subset (25% of data)
                subset_indices = random.sample(range(total_points), subset_size)
                subset = self.full_dataset[subset_indices]
                
                # Find ALL optimal k values for this subset
                valid_ks = self.find_all_k_for_subset(subset, max_k)
                
                if valid_ks:  # If any k values work
                    # Store the trial results
                    all_k_results.append({
                        'trial_idx': trial,
                        'valid_ks': valid_ks,
                        'subset': subset,
                        'subset_indices': subset_indices
                    })
                    
                    # Add all k values to the flat list
                    all_successful_ks.extend(valid_ks)
                else:
                    failed_trials += 1
                    
            except Exception as e:
                failed_trials += 1
                continue
        
        print(f"\nMonte Carlo Results:")
        print(f"  Trials with valid k values: {len(all_k_results)}")
        print(f"  Failed trials: {failed_trials}")
        print(f"  Success rate: {len(all_k_results)/n_trials*100:.1f}%")
        print(f"  Total k values found: {len(all_successful_ks)}")
        print(f"  Unique k values: {len(set(all_successful_ks))}")
        
        return all_successful_ks, all_k_results
    
    def analyze_all_k_results(self, all_successful_ks, all_k_results):
        """Analyze the comprehensive k results"""
        if not all_successful_ks:
            print("No successful k values to analyze!")
            return None, None, None
        
        # Basic statistics
        k_array = np.array(all_successful_ks)
        mean_k = np.mean(k_array)
        std_k = np.std(k_array)
        median_k = np.median(k_array)
        mode_result = Counter(all_successful_ks).most_common(1)
        mode_k = mode_result[0][0] if mode_result else None
        
        print(f"\nComprehensive K Distribution Analysis:")
        print(f"  Mean k: {mean_k:.2f}")
        print(f"  Median k: {median_k:.2f}")
        print(f"  Mode k: {mode_k}")
        print(f"  Standard deviation: {std_k:.2f}")
        print(f"  Min k: {min(all_successful_ks)}")
        print(f"  Max k: {max(all_successful_ks)}")
        
        # Analyze k ranges per trial
        k_ranges_per_trial = []
        k_counts_per_trial = []
        
        for result in all_k_results:
            valid_ks = result['valid_ks']
            if valid_ks:
                k_ranges_per_trial.append((min(valid_ks), max(valid_ks)))
                k_counts_per_trial.append(len(valid_ks))
        
        print(f"\nPer-trial Analysis:")
        print(f"  Average k values per successful trial: {np.mean(k_counts_per_trial):.2f}")
        print(f"  Max k values in a single trial: {max(k_counts_per_trial) if k_counts_per_trial else 0}")
        print(f"  Min k values in a single trial: {min(k_counts_per_trial) if k_counts_per_trial else 0}")
        
        # Create comprehensive visualization
        plt.figure(figsize=(20, 12))
        
        # Overall distribution
        plt.subplot(2, 4, 1)
        plt.hist(all_successful_ks, bins=max(20, len(set(all_successful_ks))), alpha=0.7, edgecolor='black')
        plt.axvline(mean_k, color='red', linestyle='--', label=f'Mean: {mean_k:.2f}', linewidth=2)
        plt.axvline(median_k, color='green', linestyle='--', label=f'Median: {median_k:.2f}', linewidth=2)
        if mode_k:
            plt.axvline(mode_k, color='orange', linestyle='--', label=f'Mode: {mode_k}', linewidth=2)
        plt.xlabel('k value')
        plt.ylabel('Frequency')
        plt.title('Distribution of ALL Valid k Values')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Count of each k value
        plt.subplot(2, 4, 2)
        k_counts = Counter(all_successful_ks)
        ks, counts = zip(*sorted(k_counts.items()))
        plt.bar(ks, counts, alpha=0.7)
        plt.xlabel('k value')
        plt.ylabel('Count')
        plt.title('Frequency of Each k Value')
        plt.grid(True, alpha=0.3)
        
        # Box plot
        plt.subplot(2, 4, 3)
        plt.boxplot(all_successful_ks)
        plt.ylabel('k value')
        plt.title('Box Plot of All k Values')
        plt.grid(True, alpha=0.3)
        
        # K values per trial distribution
        plt.subplot(2, 4, 4)
        plt.hist(k_counts_per_trial, bins=20, alpha=0.7, edgecolor='black')
        plt.xlabel('Number of valid k values')
        plt.ylabel('Number of trials')
        plt.title('Distribution of k Count per Trial')
        plt.grid(True, alpha=0.3)
        
        # K range visualization
        plt.subplot(2, 4, 5)
        if k_ranges_per_trial:
            min_ks, max_ks = zip(*k_ranges_per_trial)
            plt.scatter(min_ks, max_ks, alpha=0.6)
            plt.xlabel('Min k in trial')
            plt.ylabel('Max k in trial')
            plt.title('k Range per Trial')
            plt.plot([0, max(max_ks)], [0, max(max_ks)], 'r--', alpha=0.5)
            plt.grid(True, alpha=0.3)
        
        # Top k values
        plt.subplot(2, 4, 6)
        top_k_values = Counter(all_successful_ks).most_common(10)
        if top_k_values:
            ks, counts = zip(*top_k_values)
            plt.bar(range(len(ks)), counts)
            plt.xticks(range(len(ks)), ks)
            plt.xlabel('k value')
            plt.ylabel('Frequency')
            plt.title('Top 10 Most Common k Values')
            plt.grid(True, alpha=0.3)
        
        # Cumulative distribution
        plt.subplot(2, 4, 7)
        unique_ks = sorted(set(all_successful_ks))
        cumulative_counts = [sum(1 for k in all_successful_ks if k <= uk) for uk in unique_ks]
        plt.plot(unique_ks, np.array(cumulative_counts) / len(all_successful_ks) * 100)
        plt.xlabel('k value')
        plt.ylabel('Cumulative percentage')
        plt.title('Cumulative Distribution of k Values')
        plt.grid(True, alpha=0.3)
        
        # Success rate by k value
        plt.subplot(2, 4, 8)
        k_success_rate = {k: (count / len(all_k_results) * 100) for k, count in Counter(all_successful_ks).items()}
        ks_sorted = sorted(k_success_rate.keys())
        rates = [k_success_rate[k] for k in ks_sorted]
        plt.plot(ks_sorted, rates, 'o-')
        plt.xlabel('k value')
        plt.ylabel('Success rate (%)')
        plt.title('Success Rate by k Value')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return mean_k, std_k, all_k_results
    
    # Keep the original method for backward compatibility
    def find_k_for_subset(self, subset, max_k=100):
        """Find optimal k for a given subset (original method - finds first k that works)"""
        valid_ks = self.find_all_k_for_subset(subset, max_k)
        return valid_ks[0] if valid_ks else None
    
    def analyze_k_distribution(self, successful_ks):
        """Analyze the distribution of successful k values (original method)"""
        if not successful_ks:
            print("No successful k values to analyze!")
            return None, None
        
        k_array = np.array(successful_ks)
        mean_k = np.mean(k_array)
        std_k = np.std(k_array)
        median_k = np.median(k_array)
        mode_result = Counter(successful_ks).most_common(1)
        mode_k = mode_result[0][0] if mode_result else None
        
        print(f"\nK Distribution Analysis:")
        print(f"  Mean k: {mean_k:.2f}")
        print(f"  Median k: {median_k:.2f}")
        print(f"  Mode k: {mode_k}")
        print(f"  Standard deviation: {std_k:.2f}")
        print(f"  Min k: {min(successful_ks)}")
        print(f"  Max k: {max(successful_ks)}")
        
        # Create histogram
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        plt.hist(successful_ks, bins=max(20, len(set(successful_ks))), alpha=0.7, edgecolor='black')
        plt.axvline(mean_k, color='red', linestyle='--', label=f'Mean: {mean_k:.2f}', linewidth=2)
        plt.axvline(median_k, color='green', linestyle='--', label=f'Median: {median_k:.2f}', linewidth=2)
        if mode_k:
            plt.axvline(mode_k, color='orange', linestyle='--', label=f'Mode: {mode_k}', linewidth=2)
        plt.xlabel('k value')
        plt.ylabel('Frequency')
        plt.title('Distribution of Optimal k Values')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 3, 2)
        k_counts = Counter(successful_ks)
        ks, counts = zip(*sorted(k_counts.items()))
        plt.bar(ks, counts, alpha=0.7)
        plt.xlabel('k value')
        plt.ylabel('Count')
        plt.title('Count of Each k Value')
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 3, 3)
        plt.boxplot(successful_ks)
        plt.ylabel('k value')
        plt.title('Box Plot of k Values')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return mean_k, std_k
    
    def find_best_subset_for_k(self, target_k, n_attempts=100, subset_fraction=0.25):
        """Find a subset that works well with the target k value"""
        print(f"\nFinding best subset for k={target_k}:")
        
        if self.full_dataset is None:
            raise ValueError("Must run monte_carlo_k_optimization first to load full dataset")
        
        total_points = len(self.full_dataset)
        subset_size = int(total_points * subset_fraction)
        target_k_int = int(round(target_k))
        
        print(f"  Looking for subset of size {subset_size} (25% of {total_points} points)")
        
        best_subset = None
        
        for attempt in tqdm(range(n_attempts), desc="Finding best subset"):
            try:
                subset_indices = random.sample(range(total_points), subset_size)
                subset = self.full_dataset[subset_indices]
                
                # Test if this subset gives the right B0 with our target k
                found_k = self.find_k_for_subset(subset, max_k=target_k_int+5)
                
                if found_k == target_k_int:
                    best_subset = subset
                    print(f"  ✅ Found suitable subset on attempt {attempt+1}")
                    break
                    
            except Exception as e:
                continue
        
        if best_subset is None:
            print(f"  ❌ Could not find subset that works with k={target_k_int} after {n_attempts} attempts")
            print(f"     Try increasing n_attempts or adjusting target_k")
        
        return best_subset

In [ ]:
# Updated execution code to find ALL k values and save working subsets

# Initialize and load the data
optimizer = MonteCarloKNNOptimizer(
    data_root='${TDL_ROOT_DIR}/results/D1',
    true_b0=9
)

# Load the data
optimizer.load_data()

# Run a quick test with fewer trials to make sure everything works
print("=== Quick Test (100 trials) - Finding ALL k values ===")
all_successful_ks_test, all_k_results_test = optimizer.monte_carlo_k_optimization_all(
    n_trials=100,           # Small number for testing
    subset_fraction=0.25,   # Use 25% of data each time
    max_k=30,              # Don't test very high k values
    use_train_data=True    # Use training data
)

if all_successful_ks_test:
    print(f"\n✅ Quick test successful!")
    print(f"  Total k values found: {len(all_successful_ks_test)}")
    print(f"  Unique k values: {sorted(set(all_successful_ks_test))}")
    print(f"  Trials with valid k: {len(all_k_results_test)}")
    print(f"  Sample of first 10 k values: {all_successful_ks_test[:10]}")
else:
    print("❌ Quick test failed - check data loading")


In [ ]:
# Analyze the quick test results
if all_successful_ks_test:
    mean_k_test, std_k_test, _ = optimizer.analyze_all_k_results(all_successful_ks_test, all_k_results_test)
    print(f"Quick test mean k: {mean_k_test:.2f}")


In [ ]:

# Run the full Monte Carlo optimization with ALL k values
print("\n" + "="*60)
print("=== Full Monte Carlo Optimization - Finding ALL k values ===")
print("="*60)

all_successful_ks, all_k_results = optimizer.monte_carlo_k_optimization_all(
    n_trials=1000,          # Full number of trials
    subset_fraction=0.25,   # Use 25% of data each time
    max_k=50,              # Test up to k=50
    use_train_data=True
)

print(f"\nFound {len(all_successful_ks)} total k values from {len(all_k_results)} successful trials out of 1000")
print(f"Unique k values found: {sorted(set(all_successful_ks))}")


In [ ]:

# Analyze the comprehensive results
if all_successful_ks:
    mean_k, std_k, all_k_results_final = optimizer.analyze_all_k_results(all_successful_ks, all_k_results)
    
    # Print detailed summary statistics
    print(f"\n📊 COMPREHENSIVE SUMMARY RESULTS:")
    print(f"   Total k values found: {len(all_successful_ks)}")
    print(f"   Unique k values: {len(set(all_successful_ks))}")
    print(f"   Optimal k (mean): {mean_k:.2f}")
    print(f"   Standard deviation: {std_k:.2f}")
    print(f"   Successful trials: {len(all_k_results)}/1000 ({len(all_k_results)/10:.1f}%)")
    
    # Show most common k values
    k_counter = Counter(all_successful_ks)
    print(f"\n🏆 TOP 5 MOST COMMON k VALUES:")
    for i, (k, count) in enumerate(k_counter.most_common(5), 1):
        percentage = count / len(all_successful_ks) * 100
        print(f"   {i}. k={k}: {count} occurrences ({percentage:.1f}%)")
    
else:
    print("❌ No successful k values found")


In [ ]:

# Find any existing subset that works with k=14
target_k = 14
best_subset_k14 = None

if 'all_k_results' in locals() and all_k_results:
    print(f"\n🎯 Looking for existing subset that works with k = {target_k}")
    
    # Find first subset that has k=14 in its valid k list
    for result in all_k_results:
        if target_k in result['valid_ks']:
            best_subset_k14 = result['subset']
            print(f"   ✅ Found subset that works with k={target_k}")
            print(f"   Valid k values: {result['valid_ks']}")
            break
    
    if best_subset_k14 is None:
        print(f"   ❌ No existing subset works with k={target_k}")
else:
    print("❌ No Monte Carlo results available")

# Save comprehensive results including all_k_results
if 'all_k_results' in locals() and all_k_results:
        
    # Also save just the working subsets separately for easy access
    working_subsets = {
        'subsets': [result['subset'] for result in all_k_results],
        'valid_ks_per_subset': [result['valid_ks'] for result in all_k_results],
        'trial_indices': [result['trial_idx'] for result in all_k_results],
        'subset_indices': [result['subset_indices'] for result in all_k_results]
    }
    
    with open('working_subsets_25_percent.pkl', 'wb') as f:
        pickle.dump(working_subsets, f)
    
    print(f"📁 All working 25% subsets saved to: working_subsets_25_percent.pkl")
    
    
    # Save best subsets as numpy arrays for easy loading
    if 'best_subset_k14' in locals() and best_subset_k14 is not None:
        np.save('optimal_subset_k14.npy', best_subset_k14)
        print("📁 k=14 subset saved to: optimal_subset_k14.npy")
    
    if 'best_subset_mean' in locals() and best_subset_mean is not None:
        np.save('optimal_subset_mean_k.npy', best_subset_mean)
        print("📁 Mean k subset saved to: optimal_subset_mean_k.npy")
    
    print(f"\n📈 Data Summary:")
    print(f"   - {len(all_k_results)} working 25% subsets saved")
    print(f"   - Each subset has {all_k_results[0]['subset'].shape} shape")
    print(f"   - k values range from {min(all_successful_ks)} to {max(all_successful_ks)}")
    print(f"   - Most common k value: {Counter(all_successful_ks).most_common(1)[0][0]}")
    if 'best_subset_k14' in locals() and best_subset_k14 is not None:
        print(f"   - ✅ k=14 specific subset found and saved")
    else:
        print(f"   - ❌ k=14 specific subset not found")
   
else:
    print("❌ No results to save")


In [ ]:
'''
how to use the dataset
# Load your k=14 subset
data = np.load('optimal_subset_k14.npy')

# Or load with complete info
with open('k14_subset_info.pkl', 'rb') as f:
    k14_info = pickle.load(f)
    k14_data = k14_info['subset_data']
    valid_ks = k14_info['all_valid_ks']

# Load k statistics if needed
with open('k_statistics.pkl', 'rb') as f:
    stats = pickle.load(f)
'''